In [1]:
# 1. Install Unsloth and training dependencies
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers

import os
# SAFEGUARD 1: Force single GPU to prevent DDP crash on Kaggle dual-T4
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 

from datasets import load_dataset, concatenate_datasets
from unsloth import FastLanguageModel, get_chat_template
from unsloth.chat_templates import standardize_sharegpt
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from kaggle_secrets import UserSecretsClient
import torch


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-pqowbz_x/unsloth_039b334440e14f23a4c0768a94dba54f
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-pqowbz_x/unsloth_039b334440e14f23a4c0768a94dba54f
  Resolved https://github.com/unslothai/unsloth.git to commit 1c946d17a0c83e2cd9199b8fd0e61d0b6378156f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
# 2. Authenticate with Hugging Face via Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token


In [3]:
# 3. Load Phase 1 Adapter Checkpoint
max_seq_length = 4096
dtype = None # Auto-detects float16 for T4 GPU
load_in_4bit = True 

phase1_model_id = "SiddardhaShayini/Granite-Claude-Code-Phase-1"

print(f"Loading Phase 1 checkpoint from {phase1_model_id}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = phase1_model_id,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_token,
)

# Put adapter into active training mode and re-arm Unsloth gradient checkpointing
model = FastLanguageModel.for_training(model, use_gradient_checkpointing = "unsloth")

# Verify that Phase 1 LoRA weights are trainable
model.print_trainable_parameters()


Loading Phase 1 checkpoint from SiddardhaShayini/Granite-Claude-Code-Phase-1...
==((====))==  Unsloth 2026.10.3: Fast Granite patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

Unsloth: ibm-granite/granite-3.1-2b-instruct had a bad pad_token (<|end_of_text|>). Using pad_token = <fim_pad>.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


trainable params: 28,180,480 || all params: 2,561,712,128 || trainable%: 1.1001


In [4]:
# 4. Load Phase 2 Datasets
data_dir = "/kaggle/input/datasets/siddardhashayini3/granite-claude-data/"

print("Loading Phase 2 datasets...")
ds_debug = load_dataset("json", data_files=data_dir + "02_debugging.jsonl", split="train")
ds_refactor = load_dataset("json", data_files=data_dir + "03_refactoring.jsonl", split="train")

# Combine, shuffle, and clean
phase2_dataset = concatenate_datasets([ds_debug, ds_refactor]).shuffle(seed=42)
phase2_dataset = phase2_dataset.filter(lambda x: len(x.get("conversations", [])) > 0)
print(f"Total Phase 2 Examples: {len(phase2_dataset)}")


Loading Phase 2 datasets...


Filter:   0%|          | 0/4400 [00:00<?, ? examples/s]

Total Phase 2 Examples: 4400


In [5]:
# 5. Format conversations with ChatML
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
)

phase2_dataset = standardize_sharegpt(phase2_dataset)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

formatted_dataset = phase2_dataset.map(formatting_prompts_func, batched = True)


[unsloth.chat_templates|WARNING]Unsloth: Will map <|im_end|> to EOS = <|end_of_text|>.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/_unsloth_sentencepiece_temp/tokenizer_4vo07sns/tokenizer_config.json.


Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/4400 [00:00<?, ? examples/s]

Map:   0%|          | 0/4400 [00:00<?, ? examples/s]

In [6]:
# 6. Configure SFTTrainer for Phase 2
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1, # Prevents OOM spikes on 4K sequences
        gradient_accumulation_steps = 8, # Effective batch size = 8
        warmup_steps = 30,
        num_train_epochs = 1,            # 1 epoch covers all ~6K examples (~750 steps)
        learning_rate = 1.5e-4,          # Slightly reduced LR protects Phase 1 skills
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "outputs_phase2",
        save_strategy = "no",            # Saves Kaggle disk space
        report_to = "none",
    ),
)

# Clear VRAM cache before execution
torch.cuda.empty_cache()


Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/4400 [00:00<?, ? examples/s]

In [7]:
# 7. Start Training
print("Starting Phase 2 Training (Maintenance & Review)...")
trainer_stats = trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 49155}.


Starting Phase 2 Training (Maintenance & Review)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,400 | Num Epochs = 1 | Total steps = 550
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 28,180,480 of 2,561,712,128 (1.10% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,0.836967
20,0.798921
30,0.840685
40,0.833940
50,0.855233
60,0.840429
70,0.759691
80,0.773036
90,0.737857
100,0.730215


In [8]:
# 8. Push Phase 2 Checkpoint
your_hf_username = "SiddardhaShayini"
repo_name = f"{your_hf_username}/Granite-Claude-Code-Phase-2"

print(f"Pushing Phase 2 LoRA adapter to {repo_name}...")
model.push_to_hub(repo_name, token = hf_token)
tokenizer.push_to_hub(repo_name, token = hf_token)

print("✅ Phase 2 Complete and Saved to Hugging Face!")


Pushing Phase 2 LoRA adapter to SiddardhaShayini/Granite-Claude-Code-Phase-2...


README.md:   0%|          | 0.00/581 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/SiddardhaShayini/Granite-Claude-Code-Phase-2


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpk8takzzv/tokenizer_config.json.


✅ Phase 2 Complete and Saved to Hugging Face!
